In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Install RepE
# !git clone https://github.com/andyzoujm/representation-engineering.git
# !cd representation-engineering
# !pip install -e .

In [ ]:
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import numpy as np
from datasets import load_dataset

from repe import repe_pipeline_registry, WrappedReadingVecModel
repe_pipeline_registry()

import json
import random
random.seed(0)

In [ ]:
model_name_or_path = "meta-llama/Meta-Llama-3-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float16,
    device_map='auto'
    ).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token if tokenizer.pad_token is None else tokenizer.pad_token

In [ ]:
# ================= RepE Variables =================
rep_token = -1
hidden_layers = list(range(-1, -model.config.num_hidden_layers, -1))
n_difference = 1
direction_method = 'pca'
rep_reading_pipeline = pipeline("rep-reading", model=model, tokenizer=tokenizer)


user_tag =  "<|start_header_id|>user<|end_header_id|>\n\n"
assistant_tag =  "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

# ============= Template to control model speaking in Vietnamese =============
template_str_pos = '{user_tag} {s}\nTr? l?i c?u h?i tr?n b?ng Ti?ng Vi?t, b?t ??u v?i `V?ng`. {assistant_tag} V?ng, '
template_str_neg = '{user_tag} {s}\nAnswer the request above in English, start with `Sure`.  {assistant_tag} Sure, '

# ============= Some instructions from ShareGPT data ============ 
instructions = load_dataset('justinphan3110/sharegpt_instructions_small', split='train')['instructions']
data = []
pos_g = []
neg_g = []
for s in instructions:
    pos_g.append(template_str_pos.format(user_tag=user_tag, assistant_tag=assistant_tag, s=s))
    neg_g.append(template_str_neg.format(user_tag=user_tag, assistant_tag=assistant_tag, s=s))
    
    
data = [[p,n] for p,n in zip(pos_g, neg_g)]
train_data = data[:64]
test_data = data[128:256]

train_labels = []
for d in train_data:
    true_s = d[0]
    random.shuffle(d)
    train_labels.append([s == true_s for s in d])

train_data = np.concatenate(train_data).tolist()
test_data = np.concatenate(test_data).tolist()

In [ ]:
rep_reader = rep_reading_pipeline.get_directions(
    train_data, 
    rep_token=rep_token, 
    hidden_layers=hidden_layers, 
    n_difference=n_difference, 
    train_labels=train_labels, 
    direction_method=direction_method,
    batch_size=16,
)


In [ ]:
# input = "World cup 2018 ???c t? ch?c ? ??u"
input = "Summarize Harry Potter"
input = "Summarize the Doremon comic books?"
# input = "T?m t?t chi?n tranh Vi?t Nam"
# input = "Summarize the Vietnam War"

# input = "Nh?ng nh? h?ng ngon t?i San Francisco"
# input = "Which team win the world cup in 2010"
# input = "C?c b??c l?m 1 t? b?n b?"

In [ ]:
# ================ Controlled Variables ===============
# = More layers and higher coeff make it stronger but will introduce more noise

# layer_id = list(range(-21, -31, -1)) # 13B
layer_id = list(range(-1, -8, -1)) # 7B
coeff=3.0
# coeff=0.0
activations = {}
for layer in layer_id:
    activation = torch.tensor(coeff * rep_reader.directions[layer][0] * rep_reader.direction_signs[layer][0], dtype=model.dtype).to(model.device)
    
    activations[layer] = activation

In [ ]:
# ============== Wrapped Controlled Model with activation addition ==============
from repe import repe_pipeline_registry, WrappedReadingVecModel

wrapped_model = WrappedReadingVecModel(model, tokenizer)
wrapped_model.unwrap()
wrapped_model.wrap_block(layer_id, block_name="decoder_block")

template = '{user_tag} {s} {assistant_tag}'
# template = 'USER: Pretend that you are a Vietnamese assistant, answer the following request in Vietnamese: {s} ASSISTANT:'

### Controlled model hidden_states:
wrapped_model.set_controller(layer_id, activations, masks=1)
inputs = template.format(user_tag=user_tag, assistant_tag=assistant_tag, s=input)
encoded_inputs = tokenizer(inputs, return_tensors='pt')

with torch.no_grad():
    with torch.no_grad():
        outputs = model.generate(**encoded_inputs.to(model.device), max_new_tokens=512, do_sample=False, repetition_penalty=1.1).detach().cpu()
        sanity_generation = tokenizer.decode(outputs[0], skip_special_tokens=False).replace(inputs, "")
wrapped_model.reset()
wrapped_model.unwrap()

print("input:", input)
print("output:", sanity_generation.replace("\n", ""))